In [6]:
import pandas as pd, numpy as np
from pathlib import Path
from xgboost import XGBRegressor

In [7]:
project_root = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw" / "train.csv").exists()),
    Path.cwd(),
)

train_path = project_root / "data" / "raw" / "train.csv"
test_path = project_root / "data" / "raw" / "test.csv"

train = pd.read_csv(train_path, parse_dates=["date"])
test = pd.read_csv(test_path, parse_dates=["date"])


In [19]:
def make_feats(df, history, lags=None, rolling_mean_lag=None):
    df = df.copy()
    d = df["date"]

    # 基础时间特征
    df["dayofweek"] = d.dt.dayofweek
    df["month"] = d.dt.month
    df["year"] = d.dt.year
    df["day"] = d.dt.day
    df["weekofyear"] = d.dt.isocalendar().week.astype(int)
    df["is_weekend"] = (df["dayofweek"] >= 5).astype(int)

    # 兼容旧写法：make_feats(df, history, 364)
    if lags is None:
        lags = []
    elif isinstance(lags, int):
        lags = [lags]

    # 生成多个 lag
    for lag in lags:
        lag_col = f"lag_{lag}"
        h = history[["date", "store", "item", "sales"]].copy()
        h["date"] = h["date"] + pd.Timedelta(days=lag)
        h = h.rename(columns={"sales": lag_col})
        df = df.merge(h, on=["date", "store", "item"], how="left")

    # E: lag_364 的 7 天中心均值
    if rolling_mean_lag is not None:
        base_col = f"lag_{rolling_mean_lag}"
        if base_col in df.columns:
            df[f"rolling_mean_7_{base_col}"] = (
                df.groupby(["store", "item"])[base_col]
                .transform(lambda s: s.rolling(window=7, center=True, min_periods=1).mean())
            )

    return df


def smape(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    denom = (np.abs(y) + np.abs(p))
    return 100 * np.mean(np.where(denom == 0, 0, 2*np.abs(y-p)/denom))

def wape(y, p): return 100*np.abs(y-p).sum()/np.abs(y).sum()


In [23]:
# --- 基础参数 ---
params_poisson = dict(
    objective="count:poisson",
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
)

params_reg = dict(
    objective="reg:squarederror",
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
)

# --- 特征集合 ---
FEATS_364 = ["store", "item", "dayofweek", "month", "year", "day", "weekofyear", "is_weekend", "lag_364"]
FEATS_91 = ["store", "item", "dayofweek", "month", "year", "day", "weekofyear", "is_weekend", "lag_91"]
FEATS_364_91 = ["store", "item", "dayofweek", "month", "year", "day", "weekofyear", "is_weekend", "lag_364", "lag_91"]
FEATS_rolling_mean_7 = [
    "store", "item", "dayofweek", "month", "year", "day", "weekofyear", "is_weekend",
    "lag_364", "rolling_mean_7_lag_364"
]

# --- 训练验证切分 ---
tr_hist1 = train[train.date < "2017-01-01"]
val1 = train[(train.date >= "2017-01-01") & (train.date < "2017-04-01")]

tr_hist2 = train[train.date < "2016-01-01"]
val2 = train[(train.date >= "2016-01-01") & (train.date < "2016-04-01")]

# --- 场景定义 ---
scenarios = [
    {
        "name": "B-scenario 1",
        "lags": [364],
        "feature_cols": FEATS_364,
        "drop_cols": ["lag_364"],
    },
    {
        "name": "B-scenario 2",
        "lags": [91],
        "feature_cols": FEATS_91,
        "drop_cols": ["lag_91"],
    },
    {
        "name": "B-scenario 3",
        "lags": [364, 91],
        "feature_cols": FEATS_364_91,
        "drop_cols": ["lag_364", "lag_91"],
    },
    {
        "name": "B-scenario 4",
        "lags": [364],
        "rolling_mean_lag": 364,
        "feature_cols": FEATS_rolling_mean_7,
        "drop_cols": ["lag_364"],
    },
]

windows = [
    ("Window 1", tr_hist1, val1),
    ("Window 2", tr_hist2, val2),  # 复核窗口：2013–2015 训练，2016 Q1 验证
]

for win_name, tr_hist, val in windows:
    for scen in scenarios:
        name = scen["name"]
        lags = scen["lags"]
        rolling_mean_lag = scen.get("rolling_mean_lag")

        X_tr = make_feats(tr_hist, tr_hist, lags=lags, rolling_mean_lag=rolling_mean_lag)
        X_va = make_feats(val, tr_hist, lags=lags, rolling_mean_lag=rolling_mean_lag)

        X_tr = X_tr.dropna(subset=scen["drop_cols"]).copy()
        X_va = X_va.dropna(subset=scen["drop_cols"]).copy()

        for obj_name, params in [("count:poisson", params_poisson), ("reg:squarederror", params_reg)]:
            model = XGBRegressor(**params)
            model.fit(X_tr[scen["feature_cols"]], X_tr["sales"])
            pred = model.predict(X_va[scen["feature_cols"]])

            print(f"{win_name} / {name} / {obj_name}")
            print("SMAPE:", smape(X_va["sales"], pred), "WAPE:", wape(X_va["sales"], pred))

Window 1 / B-scenario 1 / count:poisson
SMAPE: 16.205411948430655 WAPE: 14.227118917192927
Window 1 / B-scenario 1 / reg:squarederror
SMAPE: 15.162516635050135 WAPE: 13.220953762948415
Window 1 / B-scenario 2 / count:poisson
SMAPE: 15.967686695395312 WAPE: 14.057977196748487
Window 1 / B-scenario 2 / reg:squarederror
SMAPE: 14.860713594480766 WAPE: 12.977205057859724
Window 1 / B-scenario 3 / count:poisson
SMAPE: 15.575406130857786 WAPE: 13.668485309677175
Window 1 / B-scenario 3 / reg:squarederror
SMAPE: 15.0459376598132 WAPE: 13.08476325374733
Window 1 / B-scenario 4 / count:poisson
SMAPE: 14.566950347873881 WAPE: 12.728077201275243
Window 1 / B-scenario 4 / reg:squarederror
SMAPE: 14.429838150888358 WAPE: 12.573555748837636
Window 2 / B-scenario 1 / count:poisson
SMAPE: 16.66389108818557 WAPE: 14.829174995513064
Window 2 / B-scenario 1 / reg:squarederror
SMAPE: 15.813842109026693 WAPE: 14.098380586984291
Window 2 / B-scenario 2 / count:poisson
SMAPE: 16.964021044029167 WAPE: 15.2047

In [ ]:
# 小实验：验证 "scenario 4 + lag_91" 和 "更宽中心平滑窗口" 是否值得保留

def add_centered_rolling(df, value_col, window, new_name=None):
    new_name = new_name or f"{value_col}_c{window}"
    df[new_name] = (
        df.groupby(["store", "item"])[value_col]
        .transform(lambda s: s.rolling(window=window, center=True, min_periods=1).mean())
    )
    return df

# 先做真正的 "scenario 4 + lag_91"
tr_hist = train[train.date < "2017-01-01"]
val = train[(train.date >= "2017-01-01") & (train.date < "2017-04-01")]

X_tr = make_feats(tr_hist, tr_hist, lags=[364, 91], rolling_mean_lag=364)
X_va = make_feats(val, tr_hist, lags=[364, 91], rolling_mean_lag=364)

X_tr = X_tr.dropna(subset=["lag_364", "lag_91"]).copy()
X_va = X_va.dropna(subset=["lag_364", "lag_91"]).copy()

FEATS_4_PLUS_91 = [
    "store", "item", "dayofweek", "month", "year", "day", "weekofyear", "is_weekend",
    "lag_364", "rolling_mean_7_lag_364", "lag_91"
]

m = XGBRegressor(**params_poisson).fit(X_tr[FEATS_4_PLUS_91], X_tr["sales"])
p = m.predict(X_va[FEATS_4_PLUS_91])

print("Scenario 4 + lag_91 | SMAPE:", smape(X_va["sales"], p), "WAPE:", wape(X_va["sales"], p))

# 小实验：尝试不同的中心平滑窗口
def add_centered_rolling(df, value_col, window, new_name=None):
    new_name = new_name or f"{value_col}_c{window}"
    df[new_name] = (
        df.groupby(["store", "item"])[value_col]
        .transform(lambda s: s.rolling(window=window, center=True, min_periods=1).mean())
    )
    return df

for w in [7, 14, 28]:
    X_tr_b = make_feats(tr_hist, tr_hist, lags=[364]).dropna(subset=["lag_364"]).copy()
    X_va_b = make_feats(val, tr_hist, lags=[364]).dropna(subset=["lag_364"]).copy()

    X_tr_b = add_centered_rolling(X_tr_b, "lag_364", w)
    X_va_b = add_centered_rolling(X_va_b, "lag_364", w)

    FEATS_B = [
        "store", "item", "dayofweek", "month", "year", "day", "weekofyear", "is_weekend",
        "lag_364", f"lag_364_c{w}"
    ]

    m_b = XGBRegressor(**params_poisson).fit(X_tr_b[FEATS_B], X_tr_b["sales"])
    p_b = m_b.predict(X_va_b[FEATS_B])
    print(f"lag_364_c{w} | SMAPE:", smape(X_va_b["sales"], p_b), "WAPE:", wape(X_va_b["sales"], p_b))

Scenario 4 + lag_91 | SMAPE: 14.45225248084941 WAPE: 12.627910815354078
lag_364_c7 | SMAPE: 14.566950347873881 WAPE: 12.728077201275243
lag_364_c14 | SMAPE: 14.18419192007449 WAPE: 12.377694848654404
lag_364_c28 | SMAPE: 13.92238928746734 WAPE: 12.111305170376552


In [28]:
# --- 回归版：尝试不同的中心平滑窗口
for win_name, tr_hist, val in windows:
    for w in [28, 56, 91]:
        X_tr_b = make_feats(tr_hist, tr_hist, lags=[364]).dropna(subset=["lag_364"]).copy()
        X_va_b = make_feats(val, tr_hist, lags=[364]).dropna(subset=["lag_364"]).copy()

        X_tr_b = add_centered_rolling(X_tr_b, "lag_364", w)
        X_va_b = add_centered_rolling(X_va_b, "lag_364", w)

        FEATS_B = [
            "store", "item", "dayofweek", "month", "year", "day", "weekofyear", "is_weekend",
            "lag_364", f"lag_364_c{w}"
        ]

        m1 = XGBRegressor(**params_poisson).fit(X_tr_b[FEATS_B], X_tr_b["sales"])
        p1 = m1.predict(X_va_b[FEATS_B])
        m2 = XGBRegressor(**params_reg).fit(X_tr_b[FEATS_B], X_tr_b["sales"])
        p2 = m2.predict(X_va_b[FEATS_B])
        print(f"lag_364_c{w} | SMAPE (Poisson):", smape(X_va_b["sales"], p1), "WAPE (Poisson):", wape(X_va_b["sales"], p1))
        print(f"lag_364_c{w} | SMAPE (Reg):", smape(X_va_b["sales"], p2), "WAPE (Reg):", wape(X_va_b["sales"], p2))


lag_364_c28 | SMAPE (Poisson): 13.92238928746734 WAPE (Poisson): 12.111305170376552
lag_364_c28 | SMAPE (Reg): 13.901517486654456 WAPE (Reg): 12.099135788022375
lag_364_c56 | SMAPE (Poisson): 13.75783354191071 WAPE (Poisson): 11.937650503381386
lag_364_c56 | SMAPE (Reg): 13.730395129322348 WAPE (Reg): 11.928904018577295
lag_364_c91 | SMAPE (Poisson): 13.775903240744828 WAPE (Poisson): 12.010504956401432
lag_364_c91 | SMAPE (Reg): 13.790823909303077 WAPE (Reg): 12.040472615055721
lag_364_c28 | SMAPE (Poisson): 14.072351087372043 WAPE (Poisson): 12.290313691851807
lag_364_c28 | SMAPE (Reg): 14.152409407146537 WAPE (Reg): 12.361289593616641
lag_364_c56 | SMAPE (Poisson): 14.103169210892396 WAPE (Poisson): 12.369382984033843
lag_364_c56 | SMAPE (Reg): 14.170816344540876 WAPE (Reg): 12.423116198752949
lag_364_c91 | SMAPE (Poisson): 14.692951796732986 WAPE (Poisson): 13.090792939582487
lag_364_c91 | SMAPE (Reg): 14.853804667922258 WAPE (Reg): 13.249608146717806


In [31]:
# 最后定方案：选poisson + lag_364_c28，再加一个c7作为辅助特征，让模型自己选择
# 最后定方案：Poisson + lag_364_c28，再用 c7 做辅助对照
def add_centered_rolling(df, value_col, window, new_name=None):
    new_name = new_name or f"{value_col}_c{window}"
    df[new_name] = (
        df.groupby(["store", "item"])[value_col]
        .transform(lambda s: s.rolling(window=window, center=True, min_periods=1).mean())
    )
    return df

# 训练/验证窗口
tr_hist = train[train.date < "2017-01-01"]
val = train[(train.date >= "2017-01-01") & (train.date < "2017-04-01")]

# 主方案：lag_364 + lag_364_c28
X_tr = make_feats(tr_hist, tr_hist, lags=[364]).dropna(subset=["lag_364"]).copy()
X_va = make_feats(val, tr_hist, lags=[364]).dropna(subset=["lag_364"]).copy()

X_tr = add_centered_rolling(X_tr, "lag_364", 28)
X_va = add_centered_rolling(X_va, "lag_364", 28)

FEATS_MAIN = [
    "store", "item", "dayofweek", "month", "year", "day", "weekofyear", "is_weekend",
    "lag_364", "lag_364_c28"
]

m_main = XGBRegressor(
    objective="count:poisson",
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
).fit(X_tr[FEATS_MAIN], X_tr["sales"])

p_main = m_main.predict(X_va[FEATS_MAIN])
print("MAIN: lag_364 + c28 | SMAPE:", smape(X_va["sales"], p_main), "WAPE:", wape(X_va["sales"], p_main))

# 对照：主方案再加 c7，看 c28 是否太粗糙
X_tr_c7 = X_tr.copy()
X_va_c7 = X_va.copy()

X_tr_c7 = add_centered_rolling(X_tr_c7, "lag_364", 7, new_name="lag_364_c7")
X_va_c7 = add_centered_rolling(X_va_c7, "lag_364", 7, new_name="lag_364_c7")

FEATS_C7 = [
    "store", "item", "dayofweek", "month", "year", "day", "weekofyear", "is_weekend",
    "lag_364", "lag_364_c28", "lag_364_c7"
]

m_c7 = XGBRegressor(
    objective="count:poisson",
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
).fit(X_tr_c7[FEATS_C7], X_tr_c7["sales"])

p_c7 = m_c7.predict(X_va_c7[FEATS_C7])
print("ADD c7: lag_364 + c28 + c7 | SMAPE:", smape(X_va_c7["sales"], p_c7), "WAPE:", wape(X_va_c7["sales"], p_c7))

MAIN: lag_364 + c28 | SMAPE: 13.92238928746734 WAPE: 12.111305170376552
ADD c7: lag_364 + c28 + c7 | SMAPE: 13.948854951444325 WAPE: 12.138288962905701


final: lag364+c28

# 全量训练 + 预测 2018

In [33]:
# 最终方案：Poisson + lag_364 + lag_364_c28

def add_centered_rolling(df, value_col, window, new_name=None):
    new_name = new_name or f"{value_col}_c{window}"
    df[new_name] = (
        df.groupby(["store", "item"])[value_col]
        .transform(lambda s: s.rolling(window=window, center=True, min_periods=1).mean())
    )
    return df

# 1) 训练集：全量训练，用 2013~2017 的 lag_364
X_all = make_feats(train, train, lags=[364]).dropna(subset=["lag_364"]).copy()
X_all = add_centered_rolling(X_all, "lag_364", 28)

# 2) 测试集：2018Q1 预测
X_test = make_feats(test, train, lags=[364]).copy()
X_test = add_centered_rolling(X_test, "lag_364", 28)

assert X_test["lag_364"].isna().sum() == 0

FEATS_FINAL = [
    "store", "item", "dayofweek", "month", "year", "day", "weekofyear", "is_weekend",
    "lag_364", "lag_364_c28"
]

model = XGBRegressor(
    objective="count:poisson",
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
).fit(X_all[FEATS_FINAL], X_all["sales"])

X_test["sales"] = np.clip(model.predict(X_test[FEATS_FINAL]), 0, None)

sub = X_test[["id", "sales"]].sort_values("id")

output_dir = project_root / "data"
output_dir.mkdir(parents=True, exist_ok=True)

sub.to_csv(output_dir / "submission.csv", index=False)

print(sub.shape)
print(sub.head())
print("min sales:", sub["sales"].min())
print("missing:", sub["sales"].isna().sum())

(45000, 2)
   id      sales
0   0  13.654970
1   1  15.401540
2   2  15.540970
3   3  16.207483
4   4  17.499025
min sales: 8.662129
missing: 0


In [35]:
import xgboost; print(xgboost.__version__)
print(params)
print(FEATS_FINAL)
print(X_all.shape, X_test.shape)
print(X_test["sales"].describe())

3.2.0
{'objective': 'reg:squarederror', 'n_estimators': 500, 'learning_rate': 0.05, 'max_depth': 6, 'subsample': 0.8, 'colsample_bytree': 0.8, 'random_state': 42}
['store', 'item', 'dayofweek', 'month', 'year', 'day', 'weekofyear', 'is_weekend', 'lag_364', 'lag_364_c28']
(731000, 12) (45000, 13)
count    45000.000000
mean        47.213188
std         23.380310
min          8.662129
25%         27.986979
50%         43.911428
75%         62.353921
max        141.730988
Name: sales, dtype: float64
